# 03 — Feature selection, logistic regression and scorecard

**Goal:** pick a small, stable, explainable set of features; fit a logistic regression on
their WoE values; turn it into a points-based scorecard; compare with a black-box challenger.

In [1]:
import os, sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pdmodel import config as C
pd.set_option("display.float_format", "{:.4f}".format)
PROC = ROOT / "data" / "processed"
from pdmodel import model
from pdmodel.validation import discrimination

In [2]:
S = pd.read_pickle(PROC / "samples.pkl")
binner = pd.read_pickle(PROC / "binner.pkl")
Xw = {k: binner.transform(v) for k, v in S.items()}
y_train = S["train"]["default"]

## 1. Filter by IV and correlation
Walk down the IV ranking; skip weak (IV < 0.02) or suspicious (IV > 0.5) features, and skip
any feature whose WoE is > 0.7 correlated with one already kept.

In [3]:
shortlist, sel_log = model.select_by_iv_and_correlation(binner, Xw["train"])
sel_log

,feature,iv,decision,reason
0,fico,0.1417,kept,
1,annual_inc,0.0934,kept,
2,loan_to_income,0.0414,kept,
3,inq_last_6mths,0.0400,kept,
4,mort_acc,0.0380,dropped,21% missing > 10%
5,purpose,0.0352,kept,
6,dti,0.0351,kept,
7,revol_util,0.0341,kept,
8,home_ownership,0.0311,kept,
9,credit_hist_years,0.0206,kept,


## 2. Backward elimination
Refit and drop features until all coefficients are **significant** and **negative**.
(Negative because higher WoE = safer = lower PD. A positive sign means collinearity
is distorting the model.)

In [4]:
final, elim_log = model.backward_eliminate(Xw["train"], y_train, shortlist)
print("final features:", final)
elim_log

final features: ['fico', 'annual_inc', 'loan_to_income', 'inq_last_6mths', 'purpose', 'dti', 'revol_util', 'home_ownership', 'credit_hist_years']


,feature,reason,coef,p_value


In [5]:
res = model.fit_logit(Xw["train"][final], y_train)
print(res.summary2().tables[0].to_string(header=False))
model.coefficient_table(res, Xw["train"][final])

0               Model:             Logit            Method:         MLE
1  Dependent Variable:                 y  Pseudo R-squared:       0.047
2                Date:  2026-10-04 21:11               AIC:  88837.4211
3    No. Observations:            122798               BIC:  88934.6041
4            Df Model:                 9    Log-Likelihood:     -44409.
5        Df Residuals:            122788           LL-Null:     -46578.
6           Converged:            1.0000       LLR p-value:      0.0000
7      No. Iterations:            7.0000             Scale:      1.0000


,coef,std_err,p_value,vif
const,-1.9379,0.0090,0.0000,NaN
fico,-0.8145,0.0289,0.0000,1.4896
annual_inc,-0.6599,0.0340,0.0000,1.3934
loan_to_income,-0.7103,0.0459,0.0000,1.1881
inq_last_6mths,-1.1055,0.0439,0.0000,1.0448
purpose,-1.0631,0.0469,0.0000,1.0090
dti,-0.4821,0.0492,0.0000,1.1398
revol_util,-0.3178,0.0579,0.0000,1.5126
home_ownership,-0.4285,0.0536,0.0000,1.1571
credit_hist_years,-0.1853,0.0651,0.0044,1.1705


## 3. Scorecard scaling
`score = offset + factor · ln(good:bad odds)`, with `factor = PDO / ln 2`.
With the defaults, **600 points = 30:1 odds, and +20 points doubles the odds**.

In [6]:
card = model.Scorecard(binner, final, res)
print(f"factor = {card.factor:.2f}, offset = {card.offset:.2f}")
pts = card.points_table()
pts

factor = 28.85, offset = 501.86


,feature,bin,woe,points,pct_of_sample,bad_rate
0,fico,"(-inf, 667]",-0.4442,52,0.1337,0.1839
1,fico,"(667, 672]",-0.3162,55,0.0748,0.1655
2,fico,"(672, 682]",-0.2088,57,0.1486,0.1512
3,fico,"(682, 687]",-0.1590,58,0.0709,0.1449
4,fico,"(687, 697]",-0.0704,60,0.1354,0.1343
...,...,...,...,...,...,...
72,credit_hist_years,"(15.09, 17.17]",0.0762,62,0.0995,0.1181
73,credit_hist_years,"(17.17, 19.83]",0.1383,63,0.0995,0.1118
74,credit_hist_years,"(19.83, 24.59]",0.1648,63,0.1004,0.1092
75,credit_hist_years,"(24.59, inf]",0.2046,63,0.0993,0.1054


Sanity check: summing the points for one applicant should reproduce `card.score()` (up to rounding).

In [7]:
one = S["test"].iloc[[0]]
manual = sum(pts.loc[(pts.feature == f) & (pts.bin == binner.bins[f].bin_labels(one[f]).iloc[0]), "points"].iloc[0]
             for f in final)
print(f"points table: {manual}   score(): {card.score(one)[0]:.1f}")

points table: 576   score(): 575.6


## 4. Challenger: gradient boosting
How much predictive power does the simple scorecard give up?

In [8]:
gbm = model.fit_challenger(S["train"])
pd.DataFrame({k: {"scorecard_gini": discrimination(v["default"], card.predict_pd(v))["gini"],
                  "challenger_gini": discrimination(v["default"], model.challenger_pd(gbm, v))["gini"]}
              for k, v in S.items()}).T

,scorecard_gini,challenger_gini
train,0.3157,0.4409
test,0.3047,0.3249
oot,0.2936,0.3164


In [9]:
pd.to_pickle({"features": final, "logit": res, "scorecard": card, "challenger": gbm}, PROC / "model.pkl")

## ✍️ Interview prep
1. Why would a bank pick the scorecard even if the challenger has a higher Gini?
2. What does the PDO parameter mean in plain English?
3. How would you explain the points table to a credit officer who doesn't know statistics?
4. What's VIF and why check it?